<a href="https://colab.research.google.com/github/deepak-kumar-da/smart-comment-classifier/blob/main/notebooks/00_dataset_research_and_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install ucimlrepo datasets openpyxl pandas numpy matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

import os

BASE = '/content/drive/MyDrive/Smart_Comment_Classifier'

FOLDERS = [
    "datasets/raw",
    "datasets/processed",
    "datasets/reports",
    "notebooks",
    "models",
    "outputs",
    "documentation"
]

for folder in FOLDERS:
  os.makedirs(os.path.join(BASE,folder),exist_ok=True)

RAW = os.path.join(BASE,"datasets/raw")
REPORTS = os.path.join(BASE,"datasets/reports")

Mounted at /content/drive


In [ ]:
# UCI Youtube Dataset

from ucimlrepo import fetch_ucirepo
import pandas as pd

uci = fetch_ucirepo(id=380)

uci_df = pd.concat(
    [
        uci.data.features.reset_index(drop=True),
        uci.data.targets.reset_index(drop=True)

    ],
    axis=1
)

uci_path = os.path.join(RAW,"uci_youtube_spam.csv")
uci_df.to_csv(uci_path,index=False)

In [ ]:
# Download Youtube 45k dataset

from datasets import load_dataset

hf_data = load_dataset(
    "atakshat11/Youtube-Comments-Dataset-45k-rows"
)

print(hf_data)

README.md:   0%|          | 0.00/3.35k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


YouTube Comments Dataset with Sentiment,(…): reconstructing file:   0%|          |  0.00B / 16.0MB            

YouTube Comments Dataset with Sentiment,(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['channel_username', 'channel_id', 'video_id', 'video_title', 'comment_text', 'comment_id', 'author', 'published_at', 'like_count', 'reply_count', 'language', 'label_sentiment', 'label_toxicity', 'label_spam'],
        num_rows: 45005
    })
})


In [ ]:
yt45_df = hf_data["train"].to_pandas()

yt45_path = os.path.join(RAW,"youtube_45k_raw.csv")
yt45_df.to_csv(yt45_path,index=False)

print("Youtube 45k shape:",yt45_df.shape)
display(yt45_df.head())

Youtube 45k shape: (45005, 14)


,channel_username,channel_id,video_id,video_title,comment_text,comment_id,author,published_at,like_count,reply_count,language,label_sentiment,label_toxicity,label_spam
0,@WWE,UCJ5v_MCY6GNUBTO8-D3XoAg,wj56SBaroCE,Jade Cargill continues to perform knowing the ...,Jade was pushed fast in another company. WWE i...,UgzBqHwpN5Vh2yrBzgx4AaABAg,@IFLDook,2025-06-24T19:52:54Z,2,0,en,positive,toxic,not spam
1,@ZeeTV,UCppHT7SZKKvar4Oc9J4oljQ,3KWNKzs_ptQ,Tum Se Tum Tak - तुम से तुम तक - 7th July Onwa...,Are ye 20 sal mushki se jiye ga tab ladki bewa...,UgwG3UA7RYn2WQG4UqR4AaABAg,@KaynatYasmin-r4l,2025-06-25T18:19:01Z,2,0,so,neutral,toxic,not spam
2,@ColorsTV,UC55IWqFLDH1Xp7iu1_xknRA,eqwlbP4f4G4,Laughter Chefs S2 | Full Episode - 41 | India ...,Aly is fcking rude for no reason its laughter ...,UgylheWfFiDbrcu_GCB4AaABAg,@Geejayy616,2025-06-24T20:18:53Z,0,0,en,neutral,toxic,not spam
3,@김프로KIMPRO,UCiVs2pnGW5mLIc1jS2nxhjg,R_gAdQqH6gA,Perfect Featuring | First Take🎬🤣,Aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa,UgwFwHS549OChJ8Undd4AaABAg,@FranciscoJavierBritogarcia,2025-06-26T05:34:30Z,0,0,so,neutral,toxic,not spam
4,@ColorsTV,UC55IWqFLDH1Xp7iu1_xknRA,g2pWwrK2fyE,Meri Bhavya Life | Ep. 55 | Uma Requests Risha...,bhavya is furfect maching rishank you don't mi...,UgyzIzjTUfhJAqx-CEh4AaABAg,@ChocoHashi,2025-06-25T04:37:32Z,0,0,en,negative,toxic,not spam


In [ ]:
# downloading jigsaw toxic comment & spam-id pair dataset

expected_files = [
    "jigsaw_toxic_train.csv",
    "uci_youtube_spam.csv",
    "youtube_45k_raw.csv",
    "spamid_pair_analyzed.csv"
]

for filename in expected_files:
  path = os.path.join(RAW,filename)
  if os.path.exists(path):
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{filename} ({size_mb:.2f} MB)")
  else:
    print(f"Missing: {filename}")

jigsaw_toxic_train.csv (65.62 MB)
uci_youtube_spam.csv (0.26 MB)
youtube_45k_raw.csv (15.23 MB)
spamid_pair_analyzed.csv (93.34 MB)


In [ ]:
# Initialize datasets as a dictionary instead of a list
datasets = {}

for filename in expected_files:
  path = os.path.join(RAW, filename)
  if os.path.exists(path):
    try:
      # Use proper separator for spamid_pair_analyzed.csv
      if filename == "spamid_pair_analyzed.csv":
        df = pd.read_csv(path, sep=';', low_memory=False, on_bad_lines='skip')
        # Drop any columns that are entirely unnamed/empty (due to trailing semicolons)
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
      else:
        df = pd.read_csv(path, low_memory=False)

      datasets[filename] = df
      print(f"\nSuccessfully read: {filename}")
      print(f"Shape:", df.shape)
      print(f"Columns:", df.columns.tolist())
    except Exception as e:
      print(f"Error reading {filename}: {e}")


Successfully read: jigsaw_toxic_train.csv
Shape: (159571, 8)
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Successfully read: uci_youtube_spam.csv
Shape: (1956, 4)
Columns: ['AUTHOR', 'DATE', 'CONTENT', 'CLASS']

Successfully read: youtube_45k_raw.csv
Shape: (45005, 14)
Columns: ['channel_username', 'channel_id', 'video_id', 'video_title', 'comment_text', 'comment_id', 'author', 'published_at', 'like_count', 'reply_count', 'language', 'label_sentiment', 'label_toxicity', 'label_spam']

Successfully read: spamid_pair_analyzed.csv
Shape: (72999, 11)
Columns: ['igid', 'comment', 'posting', 'emoji', 'spam', 'lengthcomment', 'lengthposting', 'counteremojicomment', 'counteremojicommentipostinguniq', 'counteremojiposting', 'counteremojipostinguniq']


In [ ]:
# dataset structure inspection

for name,df in datasets.items():
  print("\n" + "=" *70)
  print("Dataset: ",name)
  print("="*70)

  print("\nShape",df.shape)
  print("\nData Types",df.dtypes)
  print("\nFirst Five Records",display(df.head()))
  print("\nMissing Values",df.isnull().sum())
  print("\nDuplicate rows",df.duplicated().sum())




Dataset:  jigsaw_toxic_train.csv

Shape (159571, 8)

Data Types id               object
comment_text     object
toxic             int64
severe_toxic      int64
obscene           int64
threat            int64
insult            int64
identity_hate     int64
dtype: object


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0



First Five Records None

Missing Values id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

Duplicate rows 0

Dataset:  uci_youtube_spam.csv

Shape (1956, 4)

Data Types AUTHOR     object
DATE       object
CONTENT    object
CLASS       int64
dtype: object


,AUTHOR,DATE,CONTENT,CLASS
0,Julius NM,2013-11-07T06:20:48,"Huh, anyway check out this you[tube] channel: ...",1
1,adam riyati,2013-11-07T12:37:15,Hey guys check out my new channel and our firs...,1
2,Evgeny Murashkin,2013-11-08T17:34:21,just for test I have to say murdev.com,1
3,ElNino Melendez,2013-11-09T08:28:43,me shaking my sexy ass on my channel enjoy ^_^ ﻿,1
4,GsMega,2013-11-10T16:05:38,watch?v=vtaRGgvGtWQ Check this out .﻿,1



First Five Records None

Missing Values AUTHOR       0
DATE       245
CONTENT      0
CLASS        0
dtype: int64

Duplicate rows 16

Dataset:  youtube_45k_raw.csv

Shape (45005, 14)

Data Types channel_username    object
channel_id          object
video_id            object
video_title         object
comment_text        object
comment_id          object
author              object
published_at        object
like_count           int64
reply_count          int64
language            object
label_sentiment     object
label_toxicity      object
label_spam          object
dtype: object


,channel_username,channel_id,video_id,video_title,comment_text,comment_id,author,published_at,like_count,reply_count,language,label_sentiment,label_toxicity,label_spam
0,@WWE,UCJ5v_MCY6GNUBTO8-D3XoAg,wj56SBaroCE,Jade Cargill continues to perform knowing the ...,Jade was pushed fast in another company. WWE i...,UgzBqHwpN5Vh2yrBzgx4AaABAg,@IFLDook,2025-06-24T19:52:54Z,2,0,en,positive,toxic,not spam
1,@ZeeTV,UCppHT7SZKKvar4Oc9J4oljQ,3KWNKzs_ptQ,Tum Se Tum Tak - तुम से तुम तक - 7th July Onwa...,Are ye 20 sal mushki se jiye ga tab ladki bewa...,UgwG3UA7RYn2WQG4UqR4AaABAg,@KaynatYasmin-r4l,2025-06-25T18:19:01Z,2,0,so,neutral,toxic,not spam
2,@ColorsTV,UC55IWqFLDH1Xp7iu1_xknRA,eqwlbP4f4G4,Laughter Chefs S2 | Full Episode - 41 | India ...,Aly is fcking rude for no reason its laughter ...,UgylheWfFiDbrcu_GCB4AaABAg,@Geejayy616,2025-06-24T20:18:53Z,0,0,en,neutral,toxic,not spam
3,@김프로KIMPRO,UCiVs2pnGW5mLIc1jS2nxhjg,R_gAdQqH6gA,Perfect Featuring | First Take🎬🤣,Aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa,UgwFwHS549OChJ8Undd4AaABAg,@FranciscoJavierBritogarcia,2025-06-26T05:34:30Z,0,0,so,neutral,toxic,not spam
4,@ColorsTV,UC55IWqFLDH1Xp7iu1_xknRA,g2pWwrK2fyE,Meri Bhavya Life | Ep. 55 | Uma Requests Risha...,bhavya is furfect maching rishank you don't mi...,UgyzIzjTUfhJAqx-CEh4AaABAg,@ChocoHashi,2025-06-25T04:37:32Z,0,0,en,negative,toxic,not spam



First Five Records None

Missing Values channel_username       0
channel_id             0
video_id               0
video_title           51
comment_text           5
comment_id             0
author                 8
published_at           0
like_count             0
reply_count            0
language            4971
label_sentiment        0
label_toxicity         0
label_spam             0
dtype: int64

Duplicate rows 222

Dataset:  spamid_pair_analyzed.csv

Shape (72999, 11)

Data Types igid                                object
comment                             object
posting                             object
emoji                              float64
spam                               float64
lengthcomment                      float64
lengthposting                      float64
counteremojicomment                float64
counteremojicommentipostinguniq    float64
counteremojiposting                float64
counteremojipostinguniq            float64
dtype: object


,igid,comment,posting,emoji,spam,lengthcomment,lengthposting,counteremojicomment,counteremojicommentipostinguniq,counteremojiposting,counteremojipostinguniq
0,4934196,Rambutnya 😍,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,11.0,101.0,1.0,1.0,0.0,0.0
1,4934196,"Kereeeeen bangeeeet siiih, aku kirim voice not...",Photoshoot for US @USER Print magazine and dig...,1.0,0.0,55.0,101.0,2.0,1.0,0.0,0.0
2,4934196,Couldn't be more proud🔥. LIT!!,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,30.0,101.0,1.0,1.0,0.0,0.0
3,4934196,Ayeeeeeeeeeeeeeeeeeee🎉🎉🎉🎉,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,25.0,101.0,4.0,1.0,0.0,0.0
4,4934196,🔥😍😍😍😍,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,5.0,101.0,5.0,2.0,0.0,0.0



First Five Records None

Missing Values igid                               125
comment                            125
posting                            125
emoji                              125
spam                               125
lengthcomment                      125
lengthposting                      125
counteremojicomment                125
counteremojicommentipostinguniq    125
counteremojiposting                125
counteremojipostinguniq            125
dtype: int64

Duplicate rows 9035


In [ ]:
# automatic label column inspection

label_keywords = [
    "label","class","spam","toxic",
    "target","category","hate","insult",
    "threat","obscene"
]

for name,df in datasets.items():
  print("\n" + "=" *70)
  print("Dataset: ",name)
  print("="*70)

  for col in df.columns:
    if any(keyword in str(col).lower()
      for keyword in label_keywords):

        print(f"\nPossible label column:{col}")
        print(df[col].value_counts(dropna=False).head(25))


Dataset:  jigsaw_toxic_train.csv

Possible label column:toxic
toxic
0    144277
1     15294
Name: count, dtype: int64

Possible label column:severe_toxic
severe_toxic
0    157976
1      1595
Name: count, dtype: int64

Possible label column:obscene
obscene
0    151122
1      8449
Name: count, dtype: int64

Possible label column:threat
threat
0    159093
1       478
Name: count, dtype: int64

Possible label column:insult
insult
0    151694
1      7877
Name: count, dtype: int64

Possible label column:identity_hate
identity_hate
0    158166
1      1405
Name: count, dtype: int64

Dataset:  uci_youtube_spam.csv

Possible label column:CLASS
CLASS
1    1005
0     951
Name: count, dtype: int64

Dataset:  youtube_45k_raw.csv

Possible label column:label_sentiment
label_sentiment
neutral     22985
positive    18210
negative     3810
Name: count, dtype: int64

Possible label column:label_toxicity
label_toxicity
non-toxic    43105
toxic         1900
Name: count, dtype: int64

Possible label column

In [ ]:
# text column inspection

text_keywords = [
    "text","comment","content",
    "message","review","body",
    "post","tweet"
]

for name,df in datasets.items():
  print("\n" + "=" *70)
  print("Dataset: ",name)
  print("="*70)

  for col in df.columns:
    if any(keyword in str(col).lower()
      for keyword in text_keywords):

        print(f"\nPossible text column:{col}")

        display(df[col].dropna().astype(str).head(5))


Dataset:  jigsaw_toxic_train.csv

Possible text column:comment_text


,comment_text
0,Explanation\nWhy the edits made under my usern...
1,D'aww! He matches this background colour I'm s...
2,"Hey man, I'm really not trying to edit war. It..."
3,"""\nMore\nI can't make any real suggestions on ..."
4,"You, sir, are my hero. Any chance you remember..."



Dataset:  uci_youtube_spam.csv

Possible text column:CONTENT


,CONTENT
0,"Huh, anyway check out this you[tube] channel: ..."
1,Hey guys check out my new channel and our firs...
2,just for test I have to say murdev.com
3,me shaking my sexy ass on my channel enjoy ^_^ ﻿
4,watch?v=vtaRGgvGtWQ Check this out .﻿



Dataset:  youtube_45k_raw.csv

Possible text column:comment_text


,comment_text
0,Jade was pushed fast in another company. WWE i...
1,Are ye 20 sal mushki se jiye ga tab ladki bewa...
2,Aly is fcking rude for no reason its laughter ...
3,Aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa
4,bhavya is furfect maching rishank you don't mi...


... (14 outputs left out)


Possible text column:counteremojiposting


,counteremojiposting
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0



Possible text column:counteremojipostinguniq


,counteremojipostinguniq
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


In [ ]:
# initial audit report

import json

audit_results = []

for name,df in datasets.items():
  result = {
      "dataset":name,
      "rows":int(df.shape[0]),
      "columns_count":int(df.shape[1]),
      "columns":df.columns.tolist(),
      "missing_cells":int(df.isnull().sum().sum()),
      "duplicate_rows":int(df.duplicated().sum()),
      "memory_usage_mb":round(df.memory_usage(deep=True).sum() / (1024 **2),2)
  }

  audit_results.append(result)

result_path = os.path.join(REPORTS,"initial_four_dataset_audit.json")
with open(result_path,"w") as f:
  json.dump(audit_results,f,indent=4)

display(pd.DataFrame(audit_results))

,dataset,rows,columns_count,columns,missing_cells,duplicate_rows,memory_usage_mb
0,jigsaw_toxic_train.csv,159571,8,"[id, comment_text, toxic, severe_toxic, obscen...",0,0,106.99
1,uci_youtube_spam.csv,1956,4,"[AUTHOR, DATE, CONTENT, CLASS]",245,16,0.81
2,youtube_45k_raw.csv,45005,14,"[channel_username, channel_id, video_id, video...",5035,222,43.86
3,spamid_pair_analyzed.csv,72999,11,"[igid, comment, posting, emoji, spam, lengthco...",1375,9035,72.56
